# Fase 4: Preparación de Datos Geoespaciales

Este notebook resuelve el reto técnico de las coordenadas faltantes. Mapea los IDs de las zonas a coordenadas (centroides), filtra con el Bounding Box de NYC y genera un dataset agregado para el Heatmap.

In [1]:
import pandas as pd
import geopandas as gpd
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
print("Librerías cargadas.")

Librerías cargadas.


In [2]:
print("Cargando geometría de Taxi Zones desde archivo local...")
zones_gdf = gpd.read_file('../datos/raw/taxi_zones.geojson')

# Convertir columnas a numérico
zones_gdf['LocationID'] = pd.to_numeric(zones_gdf['LocationID'])

# Calcular centroides. 
# Reproyectamos a EPSG:2263 (Sistema Plano Estatal de NY) para que el centroide sea preciso geométricamente
# y luego volvemos a proyectar los centroides a EPSG:4326 (Lat/Lon)
zones_gdf_proj = zones_gdf.to_crs(epsg=2263)
centroids_proj = zones_gdf_proj.centroid
centroids = centroids_proj.to_crs(epsg=4326)

# Extraer Lat y Lon
zones_gdf['pickup_longitude'] = centroids.x
zones_gdf['pickup_latitude'] = centroids.y

# Crear un dataframe simple para hacer el merge
geo_lookup = zones_gdf[['LocationID', 'pickup_latitude', 'pickup_longitude']].rename(columns={'LocationID': 'PULocationID'})
print(f"Coordenadas calculadas para {len(geo_lookup)} zonas.")
display(geo_lookup.head())

Cargando geometría de Taxi Zones desde archivo local...
Coordenadas calculadas para 263 zonas.


,PULocationID,pickup_latitude,pickup_longitude
0,1,40.691830,-74.174002
1,2,40.616746,-73.831300
2,3,40.864474,-73.847422
3,4,40.723752,-73.976968
4,5,40.552659,-74.188485


In [3]:
print("Cargando dataset limpio...")
df = pd.read_parquet('../datos/procesados/yellow_tripdata_2025-01_clean.parquet')
original_len = len(df)
print(f"Dataset cargado con {original_len:,} viajes.")

print("Cruzando con coordenadas (solo origen/pickup)...")
df_geo = df.merge(geo_lookup, on='PULocationID', how='inner')
print(f"Registros retenidos tras cruce: {len(df_geo):,} ({(len(df_geo)/original_len)*100:.2f}%)")

Cargando dataset limpio...
Dataset cargado con 3,250,191 viajes.
Cruzando con coordenadas (solo origen/pickup)...
Registros retenidos tras cruce: 3,242,021 (99.75%)


In [4]:
print("Aplicando Bounding Box de NYC...")
# Requisito de la rúbrica: lat ∈ [40.4774, 40.9176], lon ∈ [-74.2591, -73.7004]
bbox_lat_min, bbox_lat_max = 40.4774, 40.9176
bbox_lon_min, bbox_lon_max = -74.2591, -73.7004

mask = (
    (df_geo['pickup_latitude'] >= bbox_lat_min) & (df_geo['pickup_latitude'] <= bbox_lat_max) &
    (df_geo['pickup_longitude'] >= bbox_lon_min) & (df_geo['pickup_longitude'] <= bbox_lon_max)
)

df_geo = df_geo[mask]
print(f"Registros dentro del Bounding Box: {len(df_geo):,} ({(len(df_geo)/original_len)*100:.2f}%)")

Aplicando Bounding Box de NYC...
Registros dentro del Bounding Box: 3,242,021 (99.75%)


In [5]:
print("Binificando coordenadas...")
# Requisito de la rúbrica: lat_bin = pickup_latitude.round(2), lon_bin = pickup_longitude.round(2)
df_geo['lat_bin'] = df_geo['pickup_latitude'].round(2)
df_geo['lon_bin'] = df_geo['pickup_longitude'].round(2)
display(df_geo[['pickup_latitude', 'lat_bin', 'pickup_longitude', 'lon_bin']].head())

Binificando coordenadas...


,pickup_latitude,lat_bin,pickup_longitude,lon_bin
0,40.756729,40.76,-73.965146,-73.97
1,40.780436,40.78,-73.957012,-73.96
2,40.766948,40.77,-73.959635,-73.96
3,40.841708,40.84,-73.941399,-73.94
4,40.841708,40.84,-73.941399,-73.94


In [6]:
print("Construyendo tabla agregada de demanda...")
# Requisito: Tabla agregada de demanda por hora × día de semana x coordenadas binificadas
# para el Dashboard (Heatmap)
demanda_agg = df_geo.groupby(['hora_dia', 'dia_semana', 'lat_bin', 'lon_bin']).size().reset_index(name='viajes')

# Visualizar la estructura de nuestra tabla optimizada
print(f"Filas en tabla agregada: {len(demanda_agg):,}")
display(demanda_agg.head())

out_path = '../datos/procesados/demanda_geo_agregada.parquet'
demanda_agg.to_parquet(out_path, index=False)
print(f"\n¡Dataset agregado guardado exitosamente en {out_path}!")
print(f"Tamaño del archivo en MB aproximado: {demanda_agg.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

Construyendo tabla agregada de demanda...
Filas en tabla agregada: 27,637


,hora_dia,dia_semana,lat_bin,lon_bin,viajes
0,0,Friday,40.57,-74.11,1
1,0,Friday,40.58,-73.96,1
2,0,Friday,40.59,-73.90,2
3,0,Friday,40.61,-73.92,3
4,0,Friday,40.62,-74.03,4



¡Dataset agregado guardado exitosamente en ../datos/procesados/demanda_geo_agregada.parquet!
Tamaño del archivo en MB aproximado: 1.14 MB


In [7]:
df = pd.read_parquet('../datos/procesados/demanda_geo_agregada.parquet')
df_lunes = df[df['dia_semana']== 'Saturday']
print(df_lunes.head(10))

     hora_dia dia_semana  lat_bin  lon_bin  viajes
300         0   Saturday    40.58   -73.99       1
301         0   Saturday    40.58   -73.96       1
302         0   Saturday    40.58   -73.94       1
303         0   Saturday    40.59   -73.98       2
304         0   Saturday    40.59   -73.94       1
305         0   Saturday    40.60   -73.98       1
306         0   Saturday    40.62   -74.03       2
307         0   Saturday    40.62   -73.96       1
308         0   Saturday    40.63   -73.93       4
309         0   Saturday    40.64   -74.00       1
